# Profiling the fractional-step solver on an A100

Before the paper compares the two formulations it has to compare them *fairly*.
The least-squares path went from 25.4 s/step to 1.62 s on an A100 through a
month of profiling; putting that beside a projection code nobody has profiled
would measure effort, not formulation.

**What a host-side run already says** (`colab/fs_profile.py`, numpy, 3 steps of
the consistent E path on the production mesh):

| phase | s/step | share | iterations/call |
|---|---|---|---|
| convective | 0.21 | 0.2 % | — |
| velocity Helmholtz | 0.67 | 0.8 % | 6 |
| **pressure (consistent E)** | **85.0** | **99.0 %** | **82** |

So the projection path is *one solve*, and everything else is noise. The
velocity Helmholtz is already excellent — 6 iterations under an FDM
preconditioner. The question for the A100 is whether that 99 % holds on a GPU,
and what can be done about it.

**Why E is hard, from its own source.** $E = G^{\mathsf T}M^{-1}G$ is a
mass-weighted Schur complement, not the assembled Laplacian $K$. A $K$-based
V-cycle preconditions it *badly* — 465 iterations against ~15 for $K$ on its own
system — which is why `lssem3d/epmg.py` builds the V-cycle on $E$ at every level.
That got it to 82. The K path, which solves $K$ instead, is 2–5× cheaper and is
the reason the two fractional-step variants differ in cost at all.


In [ ]:
#@title 1. GPU and CuPy
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('GPU:', name)
try:
    import cupy, numpy
    print('cupy', cupy.__version__, '| numpy', numpy.__version__)
except ImportError:
    print('installing cupy...')
    !pip install -q cupy-cuda12x
    import cupy; print('cupy', cupy.__version__)


In [ ]:
#@title 2. Code
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib
print('\nvendored fractional-step tree:',
      len(__import__('glob').glob('fractional_step/**/*.py', recursive=True)), 'python files')


In [ ]:
#@title 3. Phase profile — the consistent (E) path
#@markdown 20 steps from the archived stationary field, restarting exactly as the
#@markdown production statistics run does.  The first step is excluded from the
#@markdown per-step figures (kernel compilation); `wall` includes everything.
STEPS = 20   #@param {type:"integer"}
TOLP  = '1e-4'  #@param {type:"string"}
!python colab/fs_profile.py --steps {STEPS} --backend cupy --consistent --tolp {TOLP}


In [ ]:
#@title 4. Phase profile — the weak-Laplacian (K) path, for contrast
#@markdown Same driver, same field, `K` instead of `E`: the pressure solve becomes
#@markdown a Helmholtz solve on the assembled Laplacian.  The cost difference
#@markdown between these two cells IS the price of controlling the weak
#@markdown divergence, measured rather than quoted.
!python colab/fs_profile.py --steps {STEPS} --backend cupy


In [ ]:
#@title 5. Is the E solve tolerance-bound or preconditioner-bound?
#@markdown Sweeping the pressure tolerance separates the two.  If iterations fall
#@markdown roughly as log(1/tol), the preconditioner is doing its job and the
#@markdown tolerance is the lever; if they barely move, the V-cycle is the lever
#@markdown and 1e-4 is already near its stagnation plateau -- which is what the
#@markdown E-run's stall at t~16 suggests.
for tol in ('1e-3', '1e-4', '1e-5'):
    print('='*72); print('tol_p =', tol, flush=True)
    !python colab/fs_profile.py --steps 8 --backend cupy --consistent --tolp {tol} 2>&1 | tail -9


## Measured on the A100 (2026-09-15)

| | s/step | pressure it/call | ms per pressure iteration | h per eddy turnover |
|---|---|---|---|---|
| E path (consistent) | 19.96 | 88 | 75.0 | 15.84 |
| K path (weak Laplacian) | 3.21 | 20 | 50.5 | 2.55 |
| least squares, for scale | 1.62 | 84 | — | 0.56 |

**Two findings, and neither is about the formulation.**

*The solve converges; the preconditioner is weak.* The tolerance sweep gives 60,
89 and 120 iterations at $10^{-3}$, $10^{-4}$, $10^{-5}$ — a clean 30 iterations
per decade, so a per-iteration reduction of 0.926. That is geometric convergence,
not the stagnation the E-run's stall suggested. But a healthy multigrid takes 1–3
iterations per decade. The K path already achieves 20 iterations on its own
operator, so 4.5× is available here to a better V-cycle on $E$.

*The A100 is losing to the GB10 on the same code.* 19.96 s/step against 7.65,
at identical iteration counts, on a device with ~46× the fp64 throughput and
~14× the bandwidth. The measured 50–75 ms per iteration is some 400× above this
problem's memory-traffic floor. That is the signature of a host-bound loop, and
the vendored tree diagnosed exactly this for its own least-squares operator —
`lssem3d/cupy_graph.py` records one matvec costing "11.45 ms regardless of
problem size", 84 % of it dispatch — then wired the CUDA-graph fix into a check
script and never into a production solver.

**The two levers multiply**, and neither has anything to do with projection
versus least squares:

| | E path | K path |
|---|---|---|
| today | 15.84 h/turnover | 2.55 |
| with graph capture (6.2×) | 2.53 | **0.41** |
| and a V-cycle as good as K's (4.5×) | **0.57** | — |

Against the least-squares path's 0.56 h/turnover, that is the honest prospect:
**with equal optimisation effort the three land within a factor of about 1.4 of
each other.** Any comparison published before that work is a comparison of how
much attention each code has had.


In [ ]:
#@title 6. Confirm the diagnosis: is the loop host-bound or device-bound?
#@markdown `cupyx.profiler.benchmark` times the same call on CPU and GPU separately.
#@markdown A gpu/cpu ratio well below 1 means the device finishes and waits while
#@markdown Python issues the next kernel -- in which case a faster GPU buys nothing
#@markdown and graph capture buys most of the wall clock.
!python colab/fs_dispatch_probe.py --backend cupy --consistent


In [ ]:
#@title 7. Can the projection path take the least-squares time step?
#@markdown The comparison above is per turnover, which already accounts for the two
#@markdown codes running at 3.5e-4 and 8e-4.  But if the projection path is stable
#@markdown at 8e-4 its cost per turnover falls by 2.3x, so this is worth 3 minutes.
#@markdown Watch `div` and `u_tau`: a rising divergence or a blow-up means no.
for dt in ('3.5e-4', '8e-4'):
    print('='*72); print('dt =', dt, flush=True)
    !python colab/fs_profile.py --steps 8 --backend cupy --consistent --dt {dt} 2>&1 | tail -8


## What to look for, and what each outcome implies

**If the pressure solve is still ~99 % on the GPU**, the projection path has
exactly one optimisation target and the rest of the code is irrelevant to its
cost. That is a cleaner situation than the least-squares path had.

**The tolerance sweep is the diagnostic that matters.** Iterations falling like
$\log(1/\mathrm{tol})$ means the V-cycle converges properly and $10^{-4}$ is a
choice; iterations flat or erratic means the V-cycle stagnates and the run is
living on its plateau — consistent with the E-run stopping near $t\approx16$
when the solves hit the iteration cap.

**Optimisation candidates, in the order I would try them.** Each is the direct
analogue of something already measured on the least-squares side:

1. **A vertex-patch smoother on $E$ instead of Chebyshev.** $E$'s coarse
   corrections miss for the same reason pointwise relaxation misses in the
   least-squares operator at large $c$ — the smoother cannot see the modes the
   coarse space cannot represent. The patch machinery is built, condensed and
   batched in the parent repository.
2. **The coarse level.** `epmg` assembles a dense $E$ per Fourier mode and
   inverts it by eigendecomposition. On the least-squares side, dropping the
   coarse degree to $p=1$ and holding the factor in fp32 gave 13× and 2× less
   memory for 5 % more iterations, and the coarse read was half the apply.
3. **Chebyshev degree.** Free to sweep, and the cheapest thing to try first.

Whatever comes out, the honest comparison is **hours per eddy turnover**, not
per step: the two schemes run at different time steps (3.5e−4 against 8e−4), so
a per-step number flatters whichever takes fewer of them.


In [ ]:
#@title 8. Cost per eddy turnover — the comparison metric
#@markdown Per-step timings flatter whichever scheme takes fewer steps, so the
#@markdown comparison is per eddy turnover (= 1/dt steps, since delta/u_tau = 1).
#@markdown The defaults are what cells 3, 4 and the production run measured on
#@markdown this GPU; change them if your run differs.
E_SPS  = 19.96   #@param {type:"number"}
K_SPS  = 3.21    #@param {type:"number"}
FS_DT  = 3.5e-4  #@param {type:"number"}
LS_SPS = 1.62    #@param {type:"number"}
LS_DT  = 8e-4    #@param {type:"number"}
runs = [('fractional step, E (consistent)', E_SPS, FS_DT),
        ('fractional step, K (weak Lap.)',  K_SPS, FS_DT),
        ('least squares (FOSLS)',           LS_SPS, LS_DT)]
print(f'{"":34s} {"s/step":>8s} {"steps/turn":>11s} {"h/turnover":>11s}')
for lab, sps, dt in runs:
    print(f'{lab:34s} {sps:8.2f} {1/dt:11,.0f} {sps/dt/3600:11.2f}')
best = min(r[1]/r[2] for r in runs)
print()
for lab, sps, dt in runs:
    print(f'  {lab:34s} {(sps/dt)/best:6.1f}x the cheapest')
print()
#@markdown **These are not the numbers to publish.** The projection path is
#@markdown host-bound on this device (cell 6) and its V-cycle is weak (cell 5).
#@markdown Correcting for both -- graph capture 6.2x, a K-quality V-cycle 4.5x --
#@markdown is what a like-for-like comparison would look like:
print('with the two implementation levers applied (projected, not measured):')
for lab, sps, dt, g in (('fractional step, E', E_SPS, FS_DT, 6.2*4.5),
                        ('fractional step, K', K_SPS, FS_DT, 6.2),
                        ('least squares', LS_SPS, LS_DT, 1.0)):
    print(f'  {lab:22s} {sps/dt/3600/g:5.2f} h per turnover'
          + ('' if g == 1 else f'   ({g:.1f}x from graph capture / preconditioner)'))
